# Datamine Grade Estimation and Kriging Workflow

This notebook demonstrates the standard workflows for block model prototyping and grade interpolation in Datamine Studio RM using `dmstudio`.

It specifically showcases three processes that are not used in `Holes3D_Tutorial.ipynb`:
1. **`protom`** - Block model prototype creation
2. **`estima`** - Univariate grade estimation (Nearest Neighbour, Inverse Distance, Ordinary Kriging)
3. **`cokrig`** - Advanced multivariate and Ordinary Kriging estimation

In [ ]:
# Step 1: Connect to Datamine Studio RM COM Automation
import os
import shutil

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, special, agent

# Connect to the running Datamine Studio RM COM Application
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
agent.initialize_sandbox()
project_folder = oScript.ActiveProject.Folder


## Step 2: Block Model Prototyping
Before grade estimation can take place, a block model prototype must be initialized.
The **`protom`** process is used to define coordinates limits and cell sizes.
Since the drillhole coordinates in `comp_holes` range from:
- X: 5923 to 6119
- Y: 4821 to 5234
- Z: -98 to 199

We define the block model prototype to cover this volume with cell sizes 10x10x10 and origin 5900, 4800, -100:

In [ ]:
# Generate block model prototype
# We pass the interactive console answers (Mined: N, Subcells: Y, Origin: 5900, 4800, -100, Size: 10,10,10, Counts: 25,45,32) via arguments:
print("Generating block model prototype (PROTOM)...")
dm_fil.protom(
    out_o='model_proto',
    rotmod_p=0,
    arguments=" 'N' 'Y' '5900' '4800' '-100' '10' '10' '10' '25' '45' '32'"
)
print("Prototype 'model_proto' created successfully.")


## Step 3: Copy and Prepare Parameter Files
Next, we prepare the search, estimation, and variogram model files using standard tutorial resources.
For `COKRIG`, we dynamically create fields and parameter tables using `pandas` and `special.inpfil`.

In [ ]:
# Copy drillholes, ESTIMA and COKRIG parameter files from database using copy_database_files helper
agent.copy_database_files({
    '_vb_comp_holes.dmx': 'comp_holes.dmx',
    '_vb_spar.dmx': 'search_params.dmx',
    '_vb_epar.dmx': 'estimation_params.dmx',
    '_vb_vpar.dmx': 'temp_vpar.dmx'
})

print("Generating COKRIG-compatible parameter files...")

# 1. Create kriging_vmodel by adding GRADE, GRADE2, VSETNUM to temp_vpar
vpar_src = agent.read_datamine("temp_vpar.dmx")
vpar_src['GRADE'] = ['CU', 'AU', 'DENSITY']
vpar_src['GRADE2'] = ['CU', 'AU', 'DENSITY']
vpar_src['VSETNUM'] = [1, 1, 1]
vpar_src['VREFNUM'] = [1, 2, 3]
vpar_src['FIPRIM'] = 0.0
vpar_src['FISEC'] = 0.0
vpar_src['FITHIRD'] = 0.0
vpar_src.to_datamine("kriging_vmodel.dmx")

if os.path.exists("temp_vpar.dmx"):
    os.remove("temp_vpar.dmx")

# 2. Create kriging_spar with dynamic parameters
# We can load search_params locally
spar_src = agent.read_datamine("search_params.dmx")
spar_src = spar_src.iloc[:1]
# Replicate parameter row for 3 grades
spar_src = pd.concat([spar_src]*3, ignore_index=True)
spar_src['SANGLE1'] = [0, 0, 0]
spar_src['SANGLE2'] = [0, 0, 0]
spar_src['SANGLE3'] = [0, 0, 0]
spar_src['SDIST1'] = [100, 100, 100]
spar_src['SDIST2'] = [100, 100, 100]
spar_src['SDIST3'] = [10, 10, 10]
spar_src['MINNUM'] = [4, 4, 4]
spar_src['MAXNUM'] = [20, 20, 20]
spar_src['SVOLNUM'] = [1, 2, 3]
spar_src.to_datamine("kriging_spar.dmx")

# 3. Create kriging_fields with GRADE, GRADE2, SVOLNUM, VSETNUM fields and output fields mapping
fields_src = pd.DataFrame([{
    'IN_VAR': 'CU', 'OUT_VAR': 'CU_EST', 'GRADE': 'CU', 'GRADE2': 'CU', 'SVOLNUM': 1, 'VSETNUM': 1, 'OK': 1,
    'ESTIM': 'CU_EST', 'NUMOBS': 'CU_NS', 'SVOL': 'CU_SVOL', 'VAR': 'CU_VAR'
}, {
    'IN_VAR': 'AU', 'OUT_VAR': 'AU_EST', 'GRADE': 'AU', 'GRADE2': 'AU', 'SVOLNUM': 2, 'VSETNUM': 1, 'OK': 1,
    'ESTIM': 'AU_EST', 'NUMOBS': 'AU_NS', 'SVOL': 'AU_SVOL', 'VAR': 'AU_VAR'
}, {
    'IN_VAR': 'DENSITY', 'OUT_VAR': 'DE_EST', 'GRADE': 'DENSITY', 'GRADE2': 'DENSITY', 'SVOLNUM': 3, 'VSETNUM': 1, 'OK': 1,
    'ESTIM': 'DE_EST', 'NUMOBS': 'DE_NS', 'SVOL': 'DE_SVOL', 'VAR': 'DE_VAR'
}])
fields_src.to_datamine("kriging_fields.dmx")

# 4. Create kriging_epar dynamically for COKRIG (loading estimation_params and adding EREFNUM/SREFNUM/IN_VAR/OUT_VAR)
epar_src = agent.read_datamine("estimation_params.dmx")
epar_src = epar_src.iloc[:3].copy()
epar_src['IN_VAR'] = ['CU', 'AU', 'DENSITY']
epar_src['OUT_VAR'] = ['CU_EST', 'AU_EST', 'DE_EST']
epar_src['VALUE_IN'] = ['CU', 'AU', 'DENSITY']
epar_src['VALUE_OU'] = ['CU_EST', 'AU_EST', 'DE_EST']
epar_src['IMETHOD'] = [3, 3, 3] # Ordinary Kriging
epar_src['VSETNUM'] = [1, 1, 1]
epar_src['VREFNUM'] = [1, 2, 3]
epar_src['SVOLNUM'] = [1, 2, 3]
epar_src['EREFNUM'] = [1, 2, 3]
epar_src['SREFNUM'] = [1, 2, 3]
epar_src.to_datamine("kriging_epar.dmx")

print("Parameter files setup complete.")


## Step 4: Univariate Grade Estimation (ESTIMA)
The **`estima`** process interpolates grade values from drillholes into the block model prototype using methods like Inverse Distance Weighting (IDW) or Nearest Neighbour (NN).

In [ ]:
# Run univariate grade estimation using ESTIMA
# Safely clean up any existing file first
estima_file = os.path.join(project_folder, 'grade_model_estima_result.dmx')
if os.path.exists(estima_file):
    try:
        os.remove(estima_file)
    except PermissionError:
        print("Warning: grade_model_estima_result.dmx is locked by Datamine Studio RM. Please unload it in the UI or estimation will fail.")

print("Running univariate grade estimation (ESTIMA)...")
dm_cmd.estima(
    proto_i='model_proto',
    in_i='comp_holes',
    srcparm_i='search_params',
    estparm_i='estimation_params',
    vmodparm_i='kriging_vmodel',
    model_o='grade_model_estima_result',
    x_f='X',
    y_f='Y',
    z_f='Z'
)
print("ESTIMA completed successfully.")


## Step 5: Advanced Multivariate Kriging (COKRIG)
For more advanced multivariate geostatistics and parallel-processed Ordinary/Simple Kriging, the **`cokrig`** process is used.
This function is also fully wrapped inside `dmcommands`.

In [ ]:
# Run advanced Kriging using COKRIG
# Safely clean up any existing file first
cokrig_file = os.path.join(project_folder, 'grade_model_cokrig_result.dmx')
if os.path.exists(cokrig_file):
    try:
        os.remove(cokrig_file)
    except PermissionError:
        print("Warning: grade_model_cokrig_result.dmx is locked by Datamine Studio RM. Please unload it in the UI or estimation will fail.")

print("Running advanced kriging estimation (COKRIG)...")
try:
    dm_cmd.cokrig(
        samples_i='comp_holes',
        proto_i='model_proto',
        fields_i='kriging_fields',
        epar_i='kriging_epar',
        spar_i='search_params',
        vmodel_i='kriging_vmodel',
        outmodel_o='grade_model_cokrig_result',
        xpt_f='X',
        ypt_f='Y',
        zpt_f='Z'
    )
    print("COKRIG execution finished.")
except Exception as e:
    print("COKRIG execution failed with exception:", e)


## Step 6: Verify and Load Results in Pandas
Finally, we can load the newly estimated block models into pandas to verify the estimated grades.

In [ ]:
# Verify and read ESTIMA output
if os.path.exists(estima_file):
    shutil.copy(estima_file, 'grade_model_estima_temp.dmx')
    df_estima = agent.read_datamine('grade_model_estima_temp.dmx')
    os.remove('grade_model_estima_temp.dmx')
    print(f"ESTIMA block model cells: {len(df_estima)}")
    print("ESTIMA Gold (AU) Grade Summary:")
    print(df_estima[df_estima['AU'] > -99]['AU'].describe())
else:
    print("Error: grade_model_estima_result.dmx was not found!")

# Verify and read COKRIG output
if os.path.exists(cokrig_file):
    shutil.copy(cokrig_file, 'grade_model_cokrig_temp.dmx')
    df_cokrig = agent.read_datamine('grade_model_cokrig_temp.dmx')
    os.remove('grade_model_cokrig_temp.dmx')
    print(f"COKRIG block model cells: {len(df_cokrig)}")
    print("COKRIG Gold (AU_EST) Grade Summary:")
    print(df_cokrig[df_cokrig['AU_EST'] > -99]['AU_EST'].describe())
else:
    print("COKRIG output was not found. (Check if your Datamine Studio RM license includes the Advanced Estimation module).")
